# ATLAS Solar Scaling Tutorial

This notebook combines two gridded solar radiation products:

1. the **downscaled reanalysis output**;
2. the **station-based IDW output**.

The integration uses the following weighted average:

$$
\Huge
ssrd_\mathrm{integrated} = \frac{\frac{\mathrm{ssrd}_{\mathrm{era5}}}{\sigma_{\mathrm{era5}}^2} + \frac{\mathrm{ssrd}_{\mathrm{idw}}}{\sigma_{\mathrm{idw}}^2}}{\frac{1}{\sigma_{\mathrm{era5}}^2} + \frac{1}{\sigma_{\mathrm{idw}}^2}}
$$

$\sigma$ is a coefficient that measures the uncertainty of the data set. A lower and constant uncertainty characterizes the downscaled dataset (0.2), while the value of sigma in the IDW data set is proportional to the distance to the closest stations, and the base coefficient is set as 0.4

The values below can be adjusted if the project needs a different balance between the downscaled field and the station based IDW field.


The final integrated product is saved in:

```text
../data/atlas_data/{country}/
```

The notebook is designed as a step-by-step tutorial. Users should only edit the parameters in **Step 1**, then run the remaining cells in order.

## Step 1. User input parameters

Edit this cell before running the notebook.

Expected input folders:

```text
../data/downscaling/{target}/{country}/
../data/stations/{country}/idw/
../data/stations/{country}/
```

Expected output folder:

```text
../data/atlas_data/{country}/
```

In [1]:
from pathlib import Path

# ---------------------------------------------------------------------
# User parameters
# ---------------------------------------------------------------------

# Country name used in the folder and file names.
# Example: "argentina", "croatia", "kenya"
country = "argentina"

# Target product generated by the downscaling workflow.
# For this solar workflow, the target is usually "ssrd".
target = "ssrd"

# Months to process.
# Use range(1, 13) to process the full year, or a list such as [1, 2, 3].
months = range(1, 2)
month = 1
# ---------------------------------------------------------------------
# Input paths
# ---------------------------------------------------------------------

# Downscaled reanalysis data produced by the downscaling notebook.
downscaled_path = Path(f"../data/downscaled_data/{target}/{country}/")

# Station-based IDW data produced by the IDW notebook.
idw_path = Path(f"../data/stations/{country}/idw/")

# Folder containing the station metadata file.
stations_path = Path(f"../data/stations/{country}/")

# Station metadata file used to estimate the distance-based IDW uncertainty.
stations_file = stations_path / f"allstats_solar_radiation_{country}.csv"

# ---------------------------------------------------------------------
# Output path
# ---------------------------------------------------------------------

output_path = Path(f"../data/atlas_data/{country}/")
output_path.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# File naming convention
# ---------------------------------------------------------------------

downscaled_file_template = f"ssrd_downscaled_{country}_m{month}.nc"
idw_file_template = f"ssrd_idw_{country}_m{month}.nc"
output_file_template = f"ssrd_integrated_{country}_m{month}.nc"
geotiff_file_template = f"ssrd_integrated_{country}_m{month}.tif"

# Variable names inside the NetCDF files.
downscaled_variable = "ssrd_downscaled"
idw_variable = "ssrd_reshaped"
output_variable = "ssrd_integrated" 

## Step 2. Import Python libraries

Run this cell without editing it.

In [2]:
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
from scipy.spatial import distance_matrix

warnings.filterwarnings("ignore")

## Step 3. Define helper functions

These functions open the monthly datasets, compute the distance-based uncertainty for the IDW layer, combine the downscaled and IDW products, and export the result as NetCDF and GeoTIFF.

In [3]:
def build_monthly_paths(month):
    """Return input and output file paths for a given month."""
    downscaled_file = downscaled_path / downscaled_file_template.format(country=country, month=month)
    idw_file = idw_path / idw_file_template.format(country=country, month=month)
    output_file = output_path / output_file_template.format(country=country, month=month)
    geotiff_file = output_path / geotiff_file_template.format(country=country, month=month)

    return downscaled_file, idw_file, output_file, geotiff_file


def open_monthly_datasets(month):
    """Open the downscaled and IDW datasets for one month."""
    downscaled_file, idw_file, _, _ = build_monthly_paths(month)

    if not downscaled_file.exists():
        raise FileNotFoundError(f"Downscaled file not found: {downscaled_file}")

    if not idw_file.exists():
        raise FileNotFoundError(f"IDW file not found: {idw_file}")

    downscaled_ds = xr.open_dataset(downscaled_file).sortby("latitude")
    downscaled_ds = downscaled_ds.reindex(latitude=downscaled_ds.latitude[::-1])

    idw_ds = xr.open_dataset(idw_file)

    return downscaled_ds, idw_ds


def ensure_epsg4326(xds):
    """Assign EPSG:4326 spatial metadata to an xarray object."""
    xds = xds.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)

    if xds.rio.crs is None:
        xds = xds.rio.write_crs("EPSG:4326", inplace=False)

    return xds


def compute_distance_matrices(stations_df, lat_grid, lon_grid):
    """Compute distance matrices from station points to the target grid."""
    station_points = stations_df[["latitude", "longitude"]].drop_duplicates().to_numpy()
    grid_points = np.array([[lat, lon] for lat in lat_grid for lon in lon_grid])

    distances = distance_matrix(station_points, grid_points) ** 3

    return [
        distances[i].reshape(len(lat_grid), len(lon_grid))
        for i in range(len(station_points))
    ]


def compute_normalized_minimum_distance(idw_ds):
    """Estimate the normalized distance from each grid point to the nearest station."""
    if not stations_file.exists():
        raise FileNotFoundError(f"Station metadata file not found: {stations_file}")

    stations_df = pd.read_csv(stations_file)

    required_columns = {"latitude", "longitude"}
    missing_columns = required_columns.difference(stations_df.columns)

    if missing_columns:
        raise ValueError(
            f"The station file must contain these columns: {sorted(required_columns)}. "
            f"Missing columns: {sorted(missing_columns)}"
        )

    lat_grid = idw_ds.latitude.to_numpy()
    lon_grid = idw_ds.longitude.to_numpy()

    distance_matrices = compute_distance_matrices(stations_df, lat_grid, lon_grid)
    minimum_distance = np.minimum.reduce(distance_matrices)

    minimum_distance_xr = xr.DataArray(
        minimum_distance,
        coords=[("latitude", lat_grid), ("longitude", lon_grid)],
    )

    distance_range = minimum_distance_xr.max() - minimum_distance_xr.min()

    if float(distance_range) == 0:
        return xr.zeros_like(minimum_distance_xr)

    return (minimum_distance_xr - minimum_distance_xr.min()) / distance_range


def compute_uncertainties(normalized_minimum_distance):
    """Compute the uncertainty terms used to integrate downscaled and IDW products."""
    sigma2_downscaled = 0.2 ** 2
    sigma2_idw = (0.4 + normalized_minimum_distance) ** 2

    return sigma2_downscaled, sigma2_idw


def export_geotiff(dataset, geotiff_file):
    """Export the integrated product as a GeoTIFF file."""
    raster = dataset[output_variable]
    raster = raster.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
    raster = raster.rio.write_crs("EPSG:4326", inplace=False)
    raster.rio.to_raster(geotiff_file)

    return geotiff_file


def integrate_products(downscaled_ds, idw_ds, month):
    """Combine downscaled and IDW data using inverse-variance weighting."""
    downscaled_ds = ensure_epsg4326(downscaled_ds)
    idw_ds = ensure_epsg4326(idw_ds)

    normalized_minimum_distance = compute_normalized_minimum_distance(idw_ds)
    sigma2_downscaled, sigma2_idw = compute_uncertainties(normalized_minimum_distance)

    integrated = (
        (downscaled_ds[downscaled_variable] / sigma2_downscaled)
        + (idw_ds[idw_variable] / sigma2_idw)
    ) / ((1 / sigma2_downscaled) + (1 / sigma2_idw))

    integrated_ds = integrated.to_dataset(name=output_variable)

    _, _, output_file, geotiff_file = build_monthly_paths(month)

    integrated_ds.to_netcdf(output_file)
    export_geotiff(integrated_ds, geotiff_file)

    return integrated_ds, output_file, geotiff_file


def process_month(month):
    """Run the full scaling workflow for a single month."""
    downscaled_ds, idw_ds = open_monthly_datasets(month)
    integrated_ds, output_file, geotiff_file = integrate_products(downscaled_ds, idw_ds, month)

    return {
        "month": month,
        "netcdf": output_file,
        "geotiff": geotiff_file,
    }

## Step 4. Run the scaling workflow

Run this cell to process all months listed in the `months` parameter.

The workflow writes one NetCDF and one GeoTIFF file per month in:

```text
../data/atlas_data/{country}/
```

In [4]:
results = []

for month in months:
    result = process_month(month)
    results.append(result)

pd.DataFrame(results)

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


,month,netcdf,geotiff
0,1,../data/atlas_data/argentina/ssrd_integrated_a...,../data/atlas_data/argentina/ssrd_integrated_a...


## Output files

After a successful run, the output folder contains files such as:

```text
ssrd_integrated_{country}_m1.nc
ssrd_integrated_{country}_m1.tif
ssrd_integrated_{country}_m2.nc
ssrd_integrated_{country}_m2.tif
...
```

These files combine the spatial detail of the downscaled reanalysis with the station-based IDW information.